# Train a Simple Audio Recognition Model

This notebook demonstrates how to train a 20 kB [Simple Audio Recognition](https://www.tensorflow.org/tutorials/sequences/audio_recognition) model to recognize keywords in speech.

The model created in this notebook is used in the [micro_speech](https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech) example for [TensorFlow Lite for MicroControllers](https://www.tensorflow.org/lite/microcontrollers/overview).

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech/train/train_micro_speech_model.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech/train/train_micro_speech_model.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>


**Training is much faster using GPU acceleration.** Before you proceed, ensure you are using a GPU runtime by going to **Runtime -> Change runtime type** and set **Hardware accelerator: GPU**. Training 15,000 iterations will take 1.5 - 2 hours on a GPU runtime.

## Configure Defaults

**MODIFY** the following constants for your specific use case.

In [ ]:
# A comma-delimited list of the words you want to train for.
# The options are: yes,no,up,down,left,right,on,off,stop,go (you can add your own
# words as well).  Any words not listed here are treated as "unknown" and a
# silence label is added automatically.
WANTED_WORDS = "yes,no,up,down,left,right,stop,go"

# derive helper values rather than hardcoding them
LABELS = WANTED_WORDS.split(',')
N_CLASSES = len(LABELS) + 2  # +2 for 'silence' and 'unknown'
SILENT_PERCENTAGE = int(100.0 / N_CLASSES)
UNKNOWN_PERCENTAGE = SILENT_PERCENTAGE

# The number of steps and learning rates can be specified as comma-separated
# lists to define the rate at each stage. For example,
# TRAINING_STEPS=12000,3000 and LEARNING_RATE=0.001,0.0001
# will run 12,000 training loops in total, with a rate of 0.001 for the first
# 8,000, and 0.0001 for the final 3,000.
TRAINING_STEPS = "12000,3000"
LEARNING_RATE = "0.001,0.0001"

# Calculate the total number of steps, which is used to identify the checkpoint
# file name.
TOTAL_STEPS = str(sum(map(lambda string: int(string), TRAINING_STEPS.split(","))))

# Print the configuration to confirm it
print("Training these words: %s" % WANTED_WORDS)
print("Number of classes (including silence/unknown): %d" % N_CLASSES)
print("Silent/unknown percentage each = %d%%" % SILENT_PERCENTAGE)
print("Training steps in each stage: %s" % TRAINING_STEPS)
print("Learning rate in each stage: %s" % LEARNING_RATE)
print("Total number of training steps: %s" % TOTAL_STEPS)

**DO NOT MODIFY** the following constants as they include filepaths used in this notebook and data that is shared during training and inference.

In [ ]:
# Calculate the percentage of 'silence' and 'unknown' training samples required
# to ensure that we have equal number of samples for each label.  previously this
# logic only worked for two keywords; using LABELS/N_CLASSES keeps it general.
number_of_labels = len(LABELS)
number_of_total_labels = N_CLASSES  # includes silence and unknown
equal_percentage_of_training_samples = int(100.0 / number_of_total_labels)
SILENT_PERCENTAGE = equal_percentage_of_training_samples
UNKNOWN_PERCENTAGE = equal_percentage_of_training_samples

# Constants which are shared during training and inference
PREPROCESS = 'micro'
WINDOW_STRIDE = 20
MODEL_ARCHITECTURE = 'tiny_conv' # Other options include: single_fc, conv,
                      # low_latency_conv, low_latency_svdf, tiny_embedding_conv

# Constants used during training only
VERBOSITY = 'WARN'
EVAL_STEP_INTERVAL = '1000'
SAVE_STEP_INTERVAL = '1000'

# Constants for training directories and filepaths
DATASET_DIR =  'dataset/'
LOGS_DIR = 'logs/'
TRAIN_DIR = 'train/' # for training checkpoints and other files.

# Constants for inference directories and filepaths
import os
MODELS_DIR = 'models'
if not os.path.exists(MODELS_DIR):
  os.mkdir(MODELS_DIR)
MODEL_TF = os.path.join(MODELS_DIR, 'model.pb')
MODEL_TFLITE = os.path.join(MODELS_DIR, 'model.tflite')
FLOAT_MODEL_TFLITE = os.path.join(MODELS_DIR, 'float_model.tflite')
MODEL_TFLITE_MICRO = os.path.join(MODELS_DIR, 'model.cc')
SAVED_MODEL = os.path.join(MODELS_DIR, 'saved_model')

QUANT_INPUT_MIN = 0.0
QUANT_INPUT_MAX = 26.0
QUANT_INPUT_RANGE = QUANT_INPUT_MAX - QUANT_INPUT_MIN

## Setup Environment

Install Dependencies

In [ ]:
import tensorflow as tf

**DELETE** any old data from previous runs


In [ ]:
!rm -rf {DATASET_DIR} {LOGS_DIR} {TRAIN_DIR} {MODELS_DIR}

Clone the TensorFlow Github Repository, which contains the relevant code required to run this tutorial.

In [ ]:
!git clone -q --depth 1 https://github.com/tensorflow/tensorflow

Load TensorBoard to visualize the accuracy and loss as training proceeds.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOGS_DIR}

## Training

The following script downloads the dataset and begin training.

In [ ]:
!python tensorflow/tensorflow/examples/speech_commands/train.py \
--data_dir={DATASET_DIR} \
--wanted_words={WANTED_WORDS} \
--silence_percentage={SILENT_PERCENTAGE} \
--unknown_percentage={UNKNOWN_PERCENTAGE} \
--preprocess={PREPROCESS} \
--window_stride={WINDOW_STRIDE} \
--model_architecture={MODEL_ARCHITECTURE} \
--how_many_training_steps={TRAINING_STEPS} \
--learning_rate={LEARNING_RATE} \
--train_dir={TRAIN_DIR} \
--summaries_dir={LOGS_DIR} \
--verbosity={VERBOSITY} \
--eval_step_interval={EVAL_STEP_INTERVAL} \
--save_step_interval={SAVE_STEP_INTERVAL}

# Skipping the training

If you don't want to spend an hour or two training the model from scratch, you can download pretrained checkpoints by uncommenting the lines below (removing the '#'s at the start of each line) and running them.

In [ ]:
#!curl -O "https://storage.googleapis.com/download.tensorflow.org/models/tflite/speech_micro_train_2020_05_10.tgz"
#!tar xzf speech_micro_train_2020_05_10.tgz

## Generate a TensorFlow Model for Inference

Combine relevant training results (graph, weights, etc) into a single file for inference. This process is known as freezing a model and the resulting model is known as a frozen model/graph, as it cannot be further re-trained after this process.

In [ ]:
!rm -rf {SAVED_MODEL}
!python tensorflow/tensorflow/examples/speech_commands/freeze.py \
--wanted_words=$WANTED_WORDS \
--window_stride_ms=$WINDOW_STRIDE \
--preprocess=$PREPROCESS \
--model_architecture=$MODEL_ARCHITECTURE \
--start_checkpoint=$TRAIN_DIR$MODEL_ARCHITECTURE'.ckpt-'{TOTAL_STEPS} \
--save_format=saved_model \
--output_file={SAVED_MODEL}

## Generate a TensorFlow Lite Model

Convert the frozen graph into a TensorFlow Lite model, which is fully quantized for use with embedded devices.

The following cell will also print the model size, which will be under 20 kilobytes.

In [ ]:
import sys
# We add this path so we can import the speech processing modules.
sys.path.append("/content/tensorflow/tensorflow/examples/speech_commands/")
import input_data
import models
import numpy as np

In [ ]:
SAMPLE_RATE = 16000
CLIP_DURATION_MS = 1000
WINDOW_SIZE_MS = 30.0
FEATURE_BIN_COUNT = 40
BACKGROUND_FREQUENCY = 0.8
BACKGROUND_VOLUME_RANGE = 0.1
TIME_SHIFT_MS = 100.0

DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz'
VALIDATION_PERCENTAGE = 10
TESTING_PERCENTAGE = 10

In [ ]:
model_settings = models.prepare_model_settings(
    len(input_data.prepare_words_list(WANTED_WORDS.split(','))),
    SAMPLE_RATE, CLIP_DURATION_MS, WINDOW_SIZE_MS,
    WINDOW_STRIDE, FEATURE_BIN_COUNT, PREPROCESS)
audio_processor = input_data.AudioProcessor(
    DATA_URL, DATASET_DIR,
    SILENT_PERCENTAGE, UNKNOWN_PERCENTAGE,
    WANTED_WORDS.split(','), VALIDATION_PERCENTAGE,
    TESTING_PERCENTAGE, model_settings, LOGS_DIR)

In [ ]:
# create an AudioPreprocessor instance for feature generation
from tflite_micro.tensorflow.lite.micro.examples.micro_speech import \
    audio_preprocessor, evaluate
feature_params = audio_preprocessor.FeatureParams()
audio_pp = audio_preprocessor.AudioPreprocessor(feature_params)

# export float model
float_converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
float_tflite_model = float_converter.convert()
float_tflite_model_size = open(FLOAT_MODEL_TFLITE, "wb").write(float_tflite_model)
print("Float model is %d bytes" % float_tflite_model_size)

# quantize with representative dataset generated by the official preprocessor
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

def representative_dataset_gen():
    import tensorflow as _tf
    from pathlib import Path
    wav_paths = list(Path(DATASET_DIR).rglob("*.wav"))[:100]
    for p in wav_paths:
        audio = _tf.io.read_file(str(p))
        waveform, _ = _tf.audio.decode_wav(audio, desired_channels=1,
                                           desired_samples=SAMPLE_RATE)
        raw = _tf.cast(waveform * 32767, _tf.int16).numpy().flatten()
        audio_pp.samples = np.expand_dims(raw, 0)
        feats = evaluate.generate_features(audio_pp)
        flattened = feats.flatten().astype(np.float32).reshape(1, -1)
        yield [flattened]

converter.representative_dataset = representative_dataset_gen


tflite_model = converter.convert()
tflite_model_size = open(MODEL_TFLITE, "wb").write(tflite_model)
print("Quantized model is %d bytes" % tflite_model_size)

## Testing the TensorFlow Lite model's accuracy

Verify that the model we've exported is still accurate, using the TF Lite Python API and our test set.

In [ ]:
# Evaluate the exported model using the official audio preprocessing
from tflite_micro.tensorflow.lite.micro.examples.micro_speech import \
    audio_preprocessor, evaluate
from pathlib import Path

# helper to run inference and accumulate accuracy

def run_on_tests(interpreter):
    feature_params = audio_preprocessor.FeatureParams()
    audio_pp = audio_preprocessor.AudioPreprocessor(feature_params)
    tests = {
        'no': 'testdata/no_1000ms.wav',
        'yes': 'testdata/yes_1000ms.wav',
        'silence': 'testdata/silence_1000ms.wav',
        'noise': 'testdata/noise_1000ms.wav'
    }
    correct = 0
    total = 0
    for label, path in tests.items():
        audio_pp.load_samples(Path(path))
        feats = evaluate.generate_features(audio_pp)
        probs = evaluate.predict(interpreter, feats)
        pred = np.argmax(probs)
        print(f"{label} probabilities = {probs}, predicted={pred}")
        # map label to index (silence=0,unknown=1,yes=2,no=3)
        idx_map = {'silence': 0, 'unknown':1, 'yes':2, 'no':3, 'noise':0}
        if idx_map[label] == pred:
            correct += 1
        total += 1
    return correct, total

# now run for quantized model
q_interpreter = tf.lite.Interpreter(MODEL_TFLITE)
q_interpreter.allocate_tensors()
print("=== Quantized model ===")
q_correct, q_total = run_on_tests(q_interpreter)
print(f"Quantized accuracy: {q_correct}/{q_total}")

# if a float model exists, evaluate it too
try:
    f_interpreter = tf.lite.Interpreter(FLOAT_MODEL_TFLITE)
    f_interpreter.allocate_tensors()
    print("=== Float model ===")
    f_correct, f_total = run_on_tests(f_interpreter)
    print(f"Float accuracy: {f_correct}/{f_total}")
except Exception:
    pass

In [ ]:
# accuracy already printed earlier via the custom evaluation code above
# you may repeat the evaluation if desired using the same audio_preprocessor

## Generate a TensorFlow Lite for MicroControllers Model
Convert the TensorFlow Lite model into a C source file that can be loaded by TensorFlow Lite for Microcontrollers.

In [ ]:
# Install xxd if it is not available
!apt-get update && apt-get -qq install xxd
# Convert to a C source file
!xxd -i {MODEL_TFLITE} > {MODEL_TFLITE_MICRO}
# Rename the buffer and length symbols to what the C++ tests expect
!sed -i 's/unsigned char .*_model_tflite\[\]/const unsigned char g_micro_speech_quantized_model_data\[\]/' {MODEL_TFLITE_MICRO}
!sed -i 's/unsigned int .*_model_tflite_len/unsigned int g_micro_speech_quantized_model_data_len/' {MODEL_TFLITE_MICRO}

In [ ]:
# Auto-generate a corresponding micro_model_settings.h header so you don't have
# to edit it by hand.  This makes kCategoryCount/Labels match WANTED_WORDS.
labels = LABELS + ['silence', 'unknown']
kCategoryCount = len(labels)
header = f"""#ifndef TENSORFLOW_LITE_MICRO_EXAMPLES_MICRO_SPEECH_MICRO_MODEL_SETTINGS_H_
#define TENSORFLOW_LITE_MICRO_EXAMPLES_MICRO_SPEECH_MICRO_MODEL_SETTINGS_H_

constexpr int kCategoryCount = {kCategoryCount};
constexpr const char* kCategoryLabels[kCategoryCount] = {{\n"""
for lbl in labels:
    header += f'    "{lbl}",\n'
header += "};\n\n#endif  // TENSORFLOW_LITE_MICRO_EXAMPLES_MICRO_SPEECH_MICRO_MODEL_SETTINGS_H_\n"
with open('micro_model_settings.h','w') as f:
    f.write(header)
print(header)

## Deploy to a Microcontroller

Follow the instructions in the [micro_speech](https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech) README.md for [TensorFlow Lite for MicroControllers](https://www.tensorflow.org/lite/microcontrollers/overview) to deploy this model on a specific microcontroller.

**Reference Model:** If you have not modified this notebook, you can follow the instructions as is, to deploy the model. Refer to the [`micro_speech/train/models`](https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/examples/micro_speech/train/models) directory to access the models generated in this notebook.

**New Model:** this notebook now automatically emits a `micro_model_settings.h` file so you don't have to edit the header by hand.  the file is generated immediately after converting to the microcontroller C source code (see previous cell).  just copy it into your firmware tree along with the `model.cc` output from the same step.

In [ ]:
# Print the C source file
!cat {MODEL_TFLITE_MICRO}